# CPB 301365 ATM Channel — CDE 1.6 Country Breakdown

## Purpose
Provides a per-country breakdown of CDE 1.6 (Total number of customers from high-risk sanctions countries)
as requested by Ben Warren for Enterprise units.

## Output Format
| Unit | Customer_Count | Country |
|------|---------------|--------|
| 301365 | X | Country Name |

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

## Part 1: Personal Customers from High-Risk Sanctions Countries

In [ ]:
%sql
-- CDE 1.6 Part 1: Personal customers by country
select '301365' as Unit,
       Count(Distinct acc.customr_num) as Customer_Count,
       ele.country_name as Country
from ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_personal_fy25 pers ON acc.customr_num = pers.customr_num AND acc.customr_bank_num = pers.customr_bank_num AND acc.customr_type = pers.customr_type
LEFT JOIN ra_fy_2025.cif_address_FY25 adr
ON pers.customr_num = adr.customr_num
JOIN ra_fy25_view.sanctions_country_risk_rating_fy2025 ele
ON upper(trim(adr.adres_country_c)) = ele.`ISO-ALPHA3`
WHERE acc.customr_bank_num = 4
AND acc.customr_type = 0
AND acc.aplictn_id in ('ACS', 'VSA')
AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
AND pers.customr_status = '00'
AND ele.RISKRATING = 'Very High'
AND upper(trim(adr.adres_country_c)) NOT IN ('CAN','')
GROUP BY ele.country_name
ORDER BY Customer_Count DESC

## Part 2: Non-Personal Customers with Higher-Risk Legal Structures

In [ ]:
%sql
-- CDE 1.6 Part 2: Non-personal customers by country
Select '301365' as Unit,
       count(*) as Customer_Count,
       ele.country_name as Country
from (
  Select DISTINCT B.customr_num as CIF_NP_CUST_NO, C.customr_num as CIF_COMP_NP_CUST_NO, A.*,
  case
    when upper(trim(A.Business_Type)) = 'S' AND upper(trim(A.Company_Type)) = 'CC'
      then C.sedi_s001_ctry_lgly_frmd
    when upper(trim(A.Business_Type)) = 'P' AND upper(trim(A.Company_Type)) IN ('CR')
      then B.customr_inc_country_c
    when upper(trim(A.Business_Type)) = 'P' AND upper(trim(A.Company_Type)) IN ('MU')
      then C.sedi_s001_ctry_lgly_frmd
    when upper(trim(A.Business_Type)) = 'I' AND upper(trim(A.Company_Type)) IN ('CR','IA')
      then B.customr_inc_country_c
    when upper(trim(A.Business_Type)) = 'I' AND upper(trim(A.Company_Type)) IN ('FN','OT')
      then C.sedi_s001_ctry_lgly_frmd
    when upper(trim(A.Business_Type)) = 'U' AND upper(trim(A.Company_Type)) IN ('SP','PA','UA','JV','FT','OT','JW')
      then C.sedi_s001_ctry_lgly_frmd
    ELSE 'OTHER'
  END Col1
  from ra_fy25_view.TECE_Industries_Legal_Structures as A
  -- TODO: Complete the JOIN to the CIF non-personal table as B from the original notebook
  Left Join ra_fy_2025.cif_compl_npers_fy25 as C
  ON A.customr_num = C.customr_num
  Where upper(trim(B.customr_inc_country_c)) != 'CAN'
  OR upper(trim(C.sedi_s001_ctry_lgly_frmd)) != 'CAN'
) as B
JOIN ra_fy25_view.sanctions_country_risk_rating_fy2025 ele
ON upper(trim(col1)) = ele.`ISO-ALPHA3`
where col1 NOT IN ('OTHER','CAN','')
AND ele.RISKRATING = 'Very High'
GROUP BY ele.country_name
ORDER BY Customer_Count DESC